# EPInformer

In [ ]:
from torch import Tensor
import torch
import torch.nn as nn
from torch.nn import Transformer, TransformerEncoderLayer, TransformerDecoderLayer
import math
from sklearn.metrics import mean_squared_error
import copy
import numpy as np
import pandas as pd
from scipy import stats
import warnings
import torch.nn.functional as F
import torch.utils.data as data_utils
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

# ELU(x)

\begin{cases}
x,&x>0\\
e^x-1,&x\leq0
\end{cases}


## seq_256bp_encoder
256 bp 的 DNA 序列→卷积提取局部序列模式→池化压缩序列长度  
该类没有强制长度为256，后续输入2000bp也能正常进行，因为模型中只指定了每次卷积的通道数，长度只是经过四次最大池化变短（除16倍）而已


In [ ]:
class seq_256bp_encoder(nn.Module):
    def __init__(self, base_size=4, out_dim=128, conv_dim=256):
        super(seq_256bp_encoder, self).__init__()
        self.conv_dim = conv_dim
        self.out_dim = out_dim # 这里没有用上
        # 四个通道可理解为A层G层T层C层
        self.base_size = base_size
        # stem_conv的功能：将原本4层通道转为256层，再ELU激活
        # 卷积层由256个4通道1*8核组成，片段编号方向只覆盖 1，保证核只在单条片段内滑动
        # padding='same' 补0控制尺寸不变
        self.stem_conv = nn.Sequential(
            nn.Conv2d(in_channels = base_size, out_channels = self.conv_dim, kernel_size = (1, 8), stride = 1, padding='same'),
            nn.ELU(),
        )
        self.conv_tower = nn.ModuleList([]) # 存储层的列表
        conv_dim = [self.conv_dim, 128, 64, 64, 128] # [256, 128, 64, 64, 128]
        for i in range(4):
            self.conv_tower.append(nn.Sequential(
                nn.Conv2d(in_channels = conv_dim[i], out_channels=conv_dim[i+1], kernel_size=(1, 3), padding=(0, 1)),
                # 长度左右补1，核长为3，最终尺寸不变
                # 通道改变
                nn.BatchNorm2d(conv_dim[i+1]), # 将上一个卷积层输出的每个通道分别归一化
                nn.ELU(),                   
                nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2)),# 划分为1*2块，取每个块最大值，最终长度减半
            ))
            self.conv_tower.append(nn.Sequential(
                nn.Conv2d(in_channels = conv_dim[i+1], out_channels=conv_dim[i+1], kernel_size=(1, 1)),
                # 约等于一个全连接，张量形状完全不变
                nn.ELU(), 
            ))
        # conv_tower的组成：Sequential1，Sequential2，Sequential1，Sequential2......总共8个
        
    def forward(self, enhancers_input):
        if enhancers_input.shape[2] == 1: # 对应每个样本是一个单个增强子的情况
            x_enhancer = enhancers_input
        else:
            x_enhancer = enhancers_input.permute(0, 3, 1, 2).contiguous()

            x_enhancer = x_enhancer.view(x_enhancer.shape[0], x_enhancer.shape[1], x_enhancer.shape[2], 1)
            # 换轴 确保为(B, 4, N, 256)


        x_enhancer = self.stem_conv(x_enhancer) # stem_conv后为(B, 256, N, 256)
        for i in range(0, len(self.conv_tower), 2): # i=0, 2, 4, 6
            x_enhancer = self.conv_tower[i](x_enhancer) 
            # 第1次后(B, 128, N, 128)
            # 第2次后(B, 64, N, 64)
            # 第3次后(B, 64, N, 32)
            # 第4次后(B, 128, N, 16)
            x_enhancer = self.conv_tower[i+1](x_enhancer) + x_enhancer # 全连接+残差连接
            # + x_enhancer 残差连接
        return x_enhancer

## enhancer_predictor_256bp
输入一条 256 bp 的增强子序列，输出一个预测的增强子活性数值

In [ ]:
class enhancer_predictor_256bp(nn.Module):
    def __init__(self):
        super(enhancer_predictor_256bp, self).__init__()
        self.encoder = seq_256bp_encoder() # 实例化上一个特征提取模型
        # (B, 4, 1, 256) → (B, 128, 1, 16) 这里N是1，代表1个增强子
        self.embedToAct = nn.Sequential(
            nn.Flatten(start_dim=1), # batch维度展开
            #(B, 128, 1, 16) → (B, 128 * 1 * 16) = (B, 2048)
            nn.Linear(128*16, 256), # (B, 2048) → (B, 256)
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1), # 训练时随机将约 10% 的输入元素置零，降低过拟合
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1), # (B, 256) → (B, 1)得到一个增强子活性分数
        )
    def forward(self, enhancer_seq):
        if len(enhancer_seq.shape) < 4:
            enhancer_seq = enhancer_seq.unsqueeze(2) # 在索引为2的位置插入一个长度为1的维度
            # (B, 4, 256) → (B, 4, 1, 256)
        seq_embed = self.encoder(enhancer_seq) # 256bp增强子转为(B, 128, 1, 16)
        epi_out = self.embedToAct(seq_embed) # (B, 1)
        return epi_out.squeeze(-1) # 删去最后一个维度，得到(B, )活性预测值

## enhancer_classifier_256bp
如果最后返回值多加一个sigmoid，作为分类器

In [ ]:
class enhancer_classifier_256bp(nn.Module):
    def __init__(self):
        super(enhancer_classifier_256bp, self).__init__()
        self.encoder = seq_256bp_encoder()
        self.embedToAct = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(128*16, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )
    def forward(self, enhancer_seq):
        if len(enhancer_seq.shape) < 4:
            enhancer_seq = enhancer_seq.unsqueeze(2)
        seq_embed = self.encoder(enhancer_seq)
        epi_out = self.embedToAct(seq_embed)
        return torch.sigmoid(epi_out.squeeze(-1)) # sigmoid

## ContinuousValueEncoder
把(Batch, N)共Batch*N个数值都转换成 d 维的特征向量

In [ ]:
class ContinuousValueEncoder(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_value: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout) # 训练时随机将约 10% 的输入元素置零，降低过拟合
        self.linear1 = nn.Linear(1, d_model)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model) # 对(Batch, N, d)所有位置归一化
        self.max_value = max_value

    def forward(self, x: Tensor) -> Tensor:
        x = x.unsqueeze(-1) # (Batch, N) --> (Batch, N, 1)
        x = torch.clamp(x, max=self.max_value) # 限制最大值只能为max，超过值置为max
        x = self.activation(self.linear1(x))
        x = self.linear2(x)
        x = self.norm(x)
        return self.dropout(x) # (Batch, N, d)

## MHAttention_encoderLayer
输入启动子和增强子的特征向量，通过多头自注意力交换信息，再通过前馈网络加工每个区域的特征，张量形状没有改变，但每个区域的特征已经根据注意力关系进行了更新  
得到启动子学习候选增强子信息后的新特征，以及启动子与各个增强子的相关权重

### 掩码key_padding_mask
  
key_padding_mask = torch.tensor([   
    [False, False, False, True,  True ],  
    [False, False, False, False, True ]  
])  

在此总共2个样本，每个样本5个特征的batch中，不关注样本1的特征4，5，不关注样本2的特征5  

### 掩码attn_mask  

attn_mask.shape == (N, N)   

 列：    P      E1     E2   
attn_mask = torch.tensor([   
    [False, False, False],  # P 可以关注所有区域   
    [False, False, True ],  # E1 不能关注 E2   
    [False, True,  False],  # E2 不能关注 E1   
])   

不能关注都将禁止关注的位置设置为负无穷  

In [ ]:
class MHAttention_encoderLayer(nn.Module):
    def __init__(self, d_model=128, nhead=8, dropout=0.):
        # 8个注意力头，不丢弃注意力权重，每个注意力头处理128/8 = 16个维度
        super(MHAttention_encoderLayer, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        # 这里用了batch_first=True，输入为(batch_size, seq_len(N), d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # 前馈网络FFN
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        ) # 128 → 512 → 128
    # self-attention block
    def _sa_block(self, x, key_padding_mask, attn_mask):
        x, w = self.self_attn(x, x, x,
                           key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        # 从输入x线性变换生成不同的但形状与x相同的Q,K,V(自注意力)
        # 以下对于batch个(N, d)的每一个进行相同的计算
        # dk为每个注意力头处理的维度数量，dk = 16,
        # 每个注意力头都有自己的QKV矩阵, 形状为(d, dk)即(128,16)
        # 每个注意力头得到的QKV矩阵的形状为(N, d)x(d, dk) = (N, dk)
        # Qi(N, dk)x[Ki(N, dk)]转置 = (N, ddk)x(dk, N) = (N, N)注意力权重矩阵（8个）
        # 相当于是N个查询与与N个键值之间的相关性矩阵
        # 对于相关性矩阵的值，除以sqrt(dk)后统一softmax，得到注意力分数矩阵
        # 注意力分数矩阵 × V(N, dk) =(N, N)x(N, dk) = (N, dk)
        # 8个新输出值(N, dk)组合，重组为(N, d)的输出值，最终返回x就是batch个输出值
        # x (batch, N, d)
        # w (batch, N, N)为8个注意力权重矩阵叠加除8得到（平均）
        # 掩码key_padding_mask，忽略补0位置
        # 掩码attn_mask，挡住未来的位置
        return x, w
        
    def forward(self, x, enhancers_padding_mask=None, attn_mask=None):
        x2 = self.norm1(x)
        x2, attention_w = self._sa_block(x2, key_padding_mask=enhancers_padding_mask, attn_mask=attn_mask)
        x = x2 + x # 残差连接，注意力后的新特征加到原始信息上
        x2 = self.norm2(x)
        x = x + self.ff(x2) # FFN后残差连接 (batch, N, d)
        return x, attention_w

## MHAttention_encoderLayer_noLN

去掉了 LayerNorm，所以名字叫 noLN；同时调整了前馈网络的宽度，并加入 Dropout  
默认的输入维度变为(batch, N, 2048)

In [ ]:
class MHAttention_encoderLayer_noLN(nn.Module):
    def __init__(self, d_model=2048, nhead=8, dim_feedforward=256, dropout=0.1, activation=F.relu):
        # 默认输入是2048维度，来自seq_256bp_encoder提取特征再flatten
        super(MHAttention_encoderLayer_noLN, self).__init__()
        self.activation = activation
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        # 这里的self_attn加入了dropout
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
    
    # 前馈网络 2048 → 256 → 2048
    def _ff_block(self, x):
        x = self.linear2(self.dropout(self.activation(self.linear1(x))))
        return self.dropout(x)
    
    # self-attention block
    def _sa_block(self, x, key_padding_mask, attn_mask):
        x, w = self.self_attn(x, x, x,
                           key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        return x, w
    
    def forward(self, x_pe, enhancers_padding_mask=None, attn_mask=None):
        xt, attention_w = self._sa_block(x_pe, enhancers_padding_mask, attn_mask=attn_mask)
        x_pe = x_pe + xt
        x_pe = x_pe + self._ff_block(x_pe)
        return x_pe, attention_w

## 最终构建EPInformer_v2

In [ ]:
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])
# 复制出 N 个结构相同、参数相互独立的神经网络模块

In [ ]:
class EPInformer_v2(nn.Module):
    def __init__(self, base_size = 4, n_encoder=3, out_dim=128, head = 4, pre_trained_encoder= None, n_enhancer=50, device='cuda', useBN=True, usePromoterSignal=True, useFeat=True, n_extraFeat=0, useLN=True):
        # n_enhancer=50取1个启动子和候选的50个增强子
        super(EPInformer_v2, self).__init__()
        self.n_enhancer = n_enhancer
        self.out_dim = out_dim
        self.useFeat = useFeat
        self.usePromoterSignal = usePromoterSignal
        self.n_extraFeat = n_extraFeat
        self.useBN = useBN
        self.base_size = base_size
        self.useLN = useLN
        # 是否有预训练的encoder，否则使用seq_256bp_encoder
        if pre_trained_encoder is not None:
            self.seq_encoder = pre_trained_encoder
            self.name = 'EPInformerV2.preTrainedConv'
        else:
            self.seq_encoder = seq_256bp_encoder(base_size=base_size)
            self.name = 'EPInformerV2'
        self.n_encoder = n_encoder # 编码器层数3层
        self.device = device
        if useLN: # 控制LayerNorm、 FFN 宽度、Dropout
            # 显式传入了d_model=out_dim=128，输入为batch*N*128，每层4个注意力头
            self.attn_encoder = get_clones(MHAttention_encoderLayer(d_model=out_dim, nhead=head), self.n_encoder)
        else:
            self.attn_encoder = get_clones(MHAttention_encoderLayer_noLN(d_model=out_dim, nhead=head), self.n_encoder)
        attn_mask = (~np.identity(self.n_enhancer+1).astype(bool))
        # np.identity(self.n_enhancer+1)创建一个单位矩阵（对角线1，其余为0）
        # .astype(bool)将矩阵值转为bool
        # ~取反
        # 创建一个对角线为 False，其余为 True的矩阵
        attn_mask[:, 0] = False # 第一行调为True
        attn_mask[0, :] = False # 第一列调为True
        # 相当于启动子可以注意力所有，但是增强子只能关注启动子和自身
        #query \ key	P	E1	E2	E3
            #P	       允许	允许 允许 允许
            #E1	       允许	允许 禁止 禁止
            #E2	       允许	禁止 允许 禁止
            #E3	       允许	禁止 禁止 允许
        attn_mask = torch.from_numpy(attn_mask)
        attn_mask.masked_fill(attn_mask, float('-inf')) # 此行多余，因为masked_fill()不是原地操作
        self.attn_mask = attn_mask
        if self.useBN: # 使用归一化层BatchNorm2d的卷积
            # 多层空洞卷积，感受野不断扩大
            self.conv_out = nn.Sequential( # 空洞卷积,后续这里传入的为(B, 128, 51, 125)
                nn.Conv2d(in_channels = 128, out_channels=64, kernel_size=(1, 3), dilation=(1, 2)),
                # 1*3的矩形 -> 空洞dilation=(1, 2)，宽度正常，长度上每个格子间距2，卷积窗口类似：□_□_□
                # 先选1，3，5再选2，4，6......
                # 最终长度变化125 -> 121
                # (B, 64, 51, 121)
                nn.BatchNorm2d(64),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=64, kernel_size=(1, 3), dilation=(1, 4)),
                # 空洞dilation=(1, 4)
                # 先选1，5，9再选2，6，10
                # 长度变化121 -> 113
                # (B, 64, 51, 113)
                nn.BatchNorm2d(64),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=64, kernel_size=(1, 3), dilation=(1, 6)),
                # 空洞dilation=(1, 6)，长度变化113 -> 101
                # (B, 64, 51, 101)
                nn.BatchNorm2d(64),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=32, kernel_size=(1, 1)),
                # 1*1卷积，只改变通道数 (B, 32, 51, 101)
                nn.BatchNorm2d(32),
                nn.ELU(),
                nn.Linear(101, int(self.out_dim/32)), # (101,4)
                nn.ELU(),
            )
        else: # 不含归一化层
            self.conv_out = nn.Sequential(
                nn.Conv2d(in_channels = 128, out_channels=64, kernel_size=(1, 3), dilation=(1, 2)),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=64, kernel_size=(1, 3), dilation=(1, 4)),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=64, kernel_size=(1, 3), dilation=(1, 6)),
                nn.ELU(),
                nn.Conv2d(in_channels = 64, out_channels=32, kernel_size=(1, 1)),
                nn.ELU(),
                nn.Linear(101, int(self.out_dim/32)),
                nn.ELU(),
            )

        n_feat = 0 # 计数RNA信息的特征
        if self.useFeat:
            n_feat += 8
        if self.usePromoterSignal:
            n_feat += 1
        self.pToExpr = nn.Sequential( # 将最终启动子表示转化为基因表达预测值
                        nn.Linear(self.out_dim+n_feat, 128),
                        nn.ReLU(),
                        nn.Linear(128, 128),
                        nn.ReLU(),
                        nn.Linear(128, 1),
                    )
        if n_extraFeat == 0:
            n_extraFeat = n_extraFeat + 1
        self.add_pos_conv = nn.Sequential( # 进入注意力之前，用CNN加入附加增强子信息
                nn.Conv1d(in_channels = self.out_dim+n_extraFeat, out_channels=self.out_dim, kernel_size=1),
                # 注意输入是(B, 129, 51)只有三个维度，使用一维卷积Conv1d
                # 卷积核为128个长度为1的读取区间，效果类似（129，128）的全连接层
                nn.ReLU(),
                nn.Conv1d(in_channels = self.out_dim, out_channels=self.out_dim, kernel_size=1),
                nn.ReLU(),
        )

    def forward(self, pe_seq, rna_feats=None, enh_feats=None):
        # pe_seq: (batch_size, n_enhancer+1 = 51, 2000, 4)
        enhancers_padding_mask = ~(pe_seq.sum(-1).sum(-1) > 0).bool()
        # 对后两个维度求和得到(B,N)的张量，若某一条序列全为补N则会导致出现求和0，转化为bool求反，被忽略
        enhancers_padding_mask[:, 0] = False # 强制允许关注第 0 个位置
        # 即使一个样本的所有序列都是空的，也不会出现所有 key 都被屏蔽的情况
        pe_embed = self.seq_encoder(pe_seq) # (B, 51, 2000, 4)
        # 先换轴-> (B, 4, 51, 2000)再stem conv -> (B, 256, 51, 2000)再conv_tower -> (B, 128, 51, 2000/16=125)
        pe_embed = self.conv_out(pe_embed) # 空洞卷积(B, 32, 51, 4)
        pe_flatten_embed = torch.flatten(pe_embed.permute(0, 2, 1, 3), start_dim=2)
        # 先换轴(B, 51, 32, 4)，再从dim2展平(B, 51, 128)
        if enh_feats is not None: # 如果还有增强子相关的信息，比如(B, 51, 1)
            pe_flatten_embed = self.add_pos_conv(torch.concat([pe_flatten_embed, enh_feats], axis=-1).permute(0,2,1)).permute(0,2,1)
        # 先组合(B, 51, 129)再换轴(B, 129, 51)
        # 进入add_pos_conv, (B, 129, 51) -> (B, 128, 51), 再换轴(B, 51, 128)
        
        attn_list = []
        for i in range(self.n_encoder): # 3层注意力机制
            pe_flatten_embed, attn = self.attn_encoder[i](pe_flatten_embed, enhancers_padding_mask=enhancers_padding_mask, attn_mask=self.attn_mask.to(self.device))
            attn_list.append(attn.unsqueeze(0))
            # 平均后（除4）的注意力权重矩阵(batch, N, N)加一个新维度(1, batch, N, N)
        p_embed = pe_flatten_embed[:,0,:] # 从输出的新张量中取出启动子的部分 (B, 128)
        if rna_feats is not None:
            p_embed = torch.cat((p_embed, rna_feats), axis=-1) # (B, 128+RNA特征维度)
        expr_out = self.pToExpr(p_embed).squeeze(-1) # 转化成启动子对应基因的表达预测
        # p_embed为(B, 128+RNA特征维度(1 or 8)), 经过线性层和激活层转为(B, 1)为基因预测值
        return expr_out, torch.cat(attn_list)